# NeuroGolf EDA: Searching For 2 x 2 Blocks

This notebook follows `task038`, a compact search-style NeuroGolf task. The model scans a 9 x 9 grid for every 2 x 2 block made entirely of color `1`, counts the hits, and writes that count as a 1 x 5 output row.

In readable Python, the task is:

```python
p = lambda grid: [([1] * count_2x2_ones(grid) + [0] * 5)[:5]]
```

The ONNX graph does the same thing with slices, multiplications, a sum, and a small one-hot-ish output encoding.

In [ ]:
import csv
import html
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
from IPython.display import HTML, Markdown, display
from matplotlib.colors import BoundaryNorm, ListedColormap
from onnx import TensorProto, helper, numpy_helper, shape_inference

TASK_ID = "task038"


def first_existing(candidates, required=True):
    """Return the first path that exists. Also search one level under /kaggle/input."""
    expanded = []
    for candidate in candidates:
        p = Path(candidate)
        expanded.append(p)
        if not p.is_absolute():
            expanded.append(Path("/kaggle/working") / p)
            input_root = Path("/kaggle/input")
            if input_root.exists():
                expanded.extend(input_root.glob(f"*/{candidate}"))
                expanded.extend(input_root.glob(f"**/{p.name}"))
    seen = set()
    for p in expanded:
        if p in seen:
            continue
        seen.add(p)
        if p.exists():
            return p
    if required:
        checked = "\n".join(str(p) for p in expanded[:20])
        raise FileNotFoundError(f"Could not find a required file. Checked examples:\n{checked}")
    return None


TASK_PATH = first_existing([
    f"/kaggle/input/competitions/neurogolf-2026/{TASK_ID}.json",
])
ONNX_PATH = first_existing([
    f"/kaggle/input/datasets/cdeotte/neurogolf-onnx/{TASK_ID}.onnx",
])
TABLE_PATH = first_existing([
    "task_by_task_release_table.csv",
], required=False)

with TASK_PATH.open() as f:
    examples = json.load(f)
model = onnx.load(str(ONNX_PATH))

release_row = {}
if TABLE_PATH:
    with TABLE_PATH.open(newline="") as f:
        for row in csv.DictReader(f):
            if row.get("task_id") == TASK_ID:
                release_row = row
                break

display(Markdown(
    f"Loaded `{TASK_ID}` from `{TASK_PATH}` and ONNX from `{ONNX_PATH}`. "
    f"Train/test/arc-gen counts: "
    f"`{ {k: len(v) for k, v in examples.items()} }`."
))

In [ ]:
# ARC / NeuroGolf color palette. Codes 10 and 11 are notebook-only markers for
# empty one-hot cells and invalid multi-hot cells.
COLORS = np.array([
    (0, 0, 0),        # 0 black
    (30, 147, 255),   # 1 blue
    (250, 61, 49),    # 2 red
    (78, 204, 48),    # 3 green
    (255, 221, 0),    # 4 yellow
    (153, 153, 153),  # 5 gray
    (229, 59, 163),   # 6 magenta
    (255, 133, 28),   # 7 orange
    (136, 216, 241),  # 8 sky
    (147, 17, 49),    # 9 maroon
    (240, 240, 240),  # 10 no color outside the logical grid
    (146, 117, 86),   # 11 invalid multi-hot marker
], dtype=float) / 255.0
CMAP = ListedColormap(COLORS)
NORM = BoundaryNorm(np.arange(-0.5, 12.5, 1), CMAP.N)


def one_hot(grid):
    arr = np.zeros((1, 10, 30, 30), dtype=np.float32)
    if max(len(grid), len(grid[0])) > 30:
        raise ValueError("NeuroGolf grids must fit inside 30x30.")
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            arr[0, int(color), r, c] = 1.0
    return arr


def trim_grid(grid):
    out = [list(row) for row in grid]
    for row in out:
        while row and row[-1] == 10:
            row.pop()
    while out and not out[-1]:
        out.pop()
    return out


def tensor_to_grid(tensor, trim=True):
    tensor = np.asarray(tensor)
    if tensor.ndim == 3:
        tensor = tensor[None, ...]
    _, channels, height, width = tensor.shape
    hits = tensor[0] > 0.5
    grid = []
    for r in range(height):
        row = []
        for c in range(width):
            active = np.flatnonzero(hits[:, r, c])
            if len(active) == 0:
                row.append(10)
            elif len(active) == 1:
                row.append(int(active[0]))
            else:
                row.append(11)
        grid.append(row)
    return trim_grid(grid) if trim else grid


def single_channel_to_grid(tensor, height=None, width=None, active_color=1, inactive_color=0, threshold=0.5):
    arr = np.asarray(tensor)
    while arr.ndim > 2:
        arr = arr[0]
    if height is not None and width is not None:
        arr = arr[:height, :width]
    return [[active_color if float(arr[r, c]) > threshold else inactive_color
             for c in range(arr.shape[1])] for r in range(arr.shape[0])]


def draw_grid(ax, grid, title="", show_values=True):
    arr = np.asarray(grid, dtype=int)
    if arr.ndim != 2:
        raise ValueError("Expected a rectangular 2D grid.")
    h, w = arr.shape
    ax.imshow(arr, cmap=CMAP, norm=NORM, interpolation="nearest")
    ax.set_title(title, fontsize=12, pad=8)
    ax.set_xticks(np.arange(-0.5, w, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, h, 1), minor=True)
    ax.grid(which="minor", color="#333333", linewidth=1)
    ax.tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    if show_values and h * w <= 100:
        for r in range(h):
            for c in range(w):
                value = int(arr[r, c])
                if value == 10:
                    continue
                text_color = "white" if value in (0, 9) else "black"
                ax.text(c, r, str(value), ha="center", va="center", color=text_color,
                        fontsize=8.5, fontweight="bold")


def show_pairs(pairs, title):
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(8.6, max(2.6, 2.45 * n)))
    if n == 1:
        axes = np.asarray([axes])
    fig.suptitle(title, fontsize=15, y=1.01)
    for i, pair in enumerate(pairs):
        draw_grid(axes[i, 0], pair["input"], f"input {i}")
        draw_grid(axes[i, 1], pair["output"], f"output {i}")
    plt.tight_layout()
    plt.show()


def show_stage(title, description, grid, badge):
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.65), gridspec_kw={"width_ratios": [1.05, 1.45]})
    ax = axes[0]
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    box = plt.Rectangle((0.08, 0.58), 0.84, 0.24, facecolor="#f7f7f7", edgecolor="#222", linewidth=1.5)
    ax.add_patch(box)
    ax.text(0.5, 0.70, badge, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.annotate("", xy=(0.50, 0.49), xytext=(0.50, 0.58), arrowprops={"arrowstyle": "-|>", "lw": 1.5})
    ax.text(0.5, 0.35, description, ha="center", va="center", fontsize=10, wrap=True)
    draw_grid(axes[1], grid, title)
    plt.tight_layout()
    plt.show()


def show_count_stage(count_value, output_row):
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.2), gridspec_kw={"width_ratios": [1.0, 1.4]})
    axes[0].axis("off")
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].text(0.5, 0.62, str(count_value), ha="center", va="center", fontsize=52, fontweight="bold")
    axes[0].text(0.5, 0.32, "ReduceSum count", ha="center", va="center", fontsize=12)
    draw_grid(axes[1], output_row, "Count encoded as a 1 x 5 row")
    plt.tight_layout()
    plt.show()


def html_table(headers, rows):
    out = ["<table style='border-collapse:collapse;font-size:14px'>"]
    out.append("<thead><tr>" + "".join(
        f"<th style='border-bottom:2px solid #333;padding:4px 8px;text-align:left'>{html.escape(str(h))}</th>"
        for h in headers) + "</tr></thead><tbody>")
    for row in rows:
        out.append("<tr>" + "".join(
            f"<td style='border-bottom:1px solid #ddd;padding:4px 8px'>{html.escape(str(x))}</td>"
            for x in row) + "</tr>")
    out.append("</tbody></table>")
    return HTML("".join(out))

## Input/Output Pairs

The input has colors `0`, `1`, and `2`. The solver searches only for 2 x 2 blocks of color `1`. The output row has one `1` for each found block, padded with `0`s to width five.

In [ ]:
show_pairs(examples["train"], "Training pairs: count the 2 x 2 blocks of color 1")

In [ ]:
show_pairs(examples["test"], "Test pair: same search, new arrangement")

## What The ONNX Solution Is

The graph starts with NeuroGolf's standard `1 x 10 x 30 x 30` one-hot tensor. For a search task, the key move is to extract just channel `1`, then build four shifted 8 x 8 views of that mask:

- `tl`: possible top-left corners of 2 x 2 windows.
- `tr`: possible top-right corners.
- `bl`: possible bottom-left corners.
- `br`: possible bottom-right corners.

Multiplying these four maps together is an AND test: a location survives only when all four cells in that 2 x 2 window are color `1`. Then `ReduceSum` counts the surviving hits.

In [ ]:
node_rows = []
for i, node in enumerate(model.graph.node):
    attrs = []
    for attr in node.attribute:
        value = helper.get_attribute_value(attr)
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        if node.op_type == "Constant" and attr.name == "value":
            arr = numpy_helper.to_array(value)
            value = arr.tolist()
        attrs.append(f"{attr.name}={value}")
    node_rows.append([
        i,
        node.op_type,
        ", ".join(node.input) or "-",
        ", ".join(node.output) or "-",
        "; ".join(attrs) or "-",
    ])

display(html_table(["#", "op", "inputs", "outputs", "attributes"], node_rows))

display(Markdown(
    "**Readable search idea:**\n\n"
    "```python\n"
    "hits = sum(grid[r][c] == grid[r][c+1] == grid[r+1][c] == grid[r+1][c+1] == 1\n"
    "           for r in range(8) for c in range(8))\n"
    "answer = [([1] * hits + [0] * 5)[:5]]\n"
    "```"
))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.6))
ax.axis("off")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
steps = [
    ("input", "9 x 9 grid"),
    ("Slice", "channel 1 mask"),
    ("Slice", "four corners"),
    ("Mul", "top pair"),
    ("Mul", "bottom pair"),
    ("Mul", "2 x 2 hits"),
    ("ReduceSum", "count"),
    ("Relu", "row encode"),
    ("Pad", "output"),
]
xs = np.linspace(0.055, 0.945, len(steps))
for i, ((name, sub), x) in enumerate(zip(steps, xs)):
    rect = plt.Rectangle((x - 0.045, 0.42), 0.09, 0.26, facecolor="#f8fbff", edgecolor="#1f2937", linewidth=1.3)
    ax.add_patch(rect)
    ax.text(x, 0.58, name, ha="center", va="center", fontsize=9.5, fontweight="bold")
    ax.text(x, 0.47, sub, ha="center", va="center", fontsize=7.5)
    if i < len(steps) - 1:
        ax.annotate("", xy=(xs[i + 1] - 0.052, 0.55), xytext=(x + 0.052, 0.55),
                    arrowprops={"arrowstyle": "-|>", "lw": 1.2, "color": "#1f2937"})
ax.set_title("ONNX graph as a sliding-window search", fontsize=15, pad=12)
plt.show()

## Layer-By-Layer Movie Strip

The next cells use one training example with four matches. Constants are skipped as images; they are slice axes, starts, ends, thresholds, and a zero initializer.

In [ ]:
selected = examples["train"][1]
input_tensor = one_hot(selected["input"])

activation_names = [
    "c1", "tl", "tr", "bl", "br", "tltr", "blbr", "block",
    "count", "count5", "c1_vals", "bg_vals", "out_1x5", "output",
]

# Add internal tensors as graph outputs so ONNX Runtime can return them.
instrumented = onnx.load(str(ONNX_PATH))
existing_outputs = {output.name for output in instrumented.graph.output}
for output_name in activation_names:
    if output_name not in existing_outputs:
        instrumented.graph.output.append(helper.make_tensor_value_info(output_name, TensorProto.FLOAT, None))
        existing_outputs.add(output_name)

session_options = ort.SessionOptions()
session_options.log_severity_level = 3
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
session = ort.InferenceSession(
    instrumented.SerializeToString(),
    sess_options=session_options,
    providers=["CPUExecutionProvider"],
)
activation_values = session.run(activation_names, {"input": input_tensor})
activations = dict(zip(activation_names, activation_values))
final_grid = tensor_to_grid(activations["output"])
count_value = int(round(float(np.asarray(activations["count"]).reshape(-1)[0])))
hit_positions = np.argwhere(np.asarray(activations["block"])[0, 0] > 0.5)

assert final_grid == selected["output"]
display(Markdown(
    f"The instrumented ONNX run found `{count_value}` windows and matches the expected output. "
    f"Hit window top-left positions: `{hit_positions.tolist()}`."
))

In [ ]:
show_stage(
    "Step 0: Input",
    "The original 9 x 9 grid contains several small islands of color 1 and color 2.",
    selected["input"],
    "input grid",
)

In [ ]:
show_stage(
    "Step 1: Channel-1 Mask",
    "Slice keeps only the color-1 channel. Blue cells are places worth searching.",
    single_channel_to_grid(activations["c1"], height=9, width=9),
    "Slice",
)

In [ ]:
show_stage(
    "Step 2: Top-Left Corner Map",
    "This 8 x 8 view asks: is the top-left cell of each possible 2 x 2 window color 1?",
    single_channel_to_grid(activations["tl"], height=8, width=8),
    "Slice",
)

In [ ]:
show_stage(
    "Step 3: Top-Right Corner Map",
    "Shift one column right to test the top-right cell of every possible window.",
    single_channel_to_grid(activations["tr"], height=8, width=8),
    "Slice",
)

In [ ]:
show_stage(
    "Step 4: Bottom-Left Corner Map",
    "Shift one row down to test the bottom-left cell of every possible window.",
    single_channel_to_grid(activations["bl"], height=8, width=8),
    "Slice",
)

In [ ]:
show_stage(
    "Step 5: Bottom-Right Corner Map",
    "Shift row and column together to test the bottom-right cell.",
    single_channel_to_grid(activations["br"], height=8, width=8),
    "Slice",
)

In [ ]:
show_stage(
    "Step 6: Top Pair Candidates",
    "Mul is an AND: a cell remains blue only when both top corners are color 1.",
    single_channel_to_grid(activations["tltr"], height=8, width=8),
    "Mul",
)

In [ ]:
show_stage(
    "Step 7: Bottom Pair Candidates",
    "The second Mul does the same AND test for the two bottom corners.",
    single_channel_to_grid(activations["blbr"], height=8, width=8),
    "Mul",
)

In [ ]:
show_stage(
    "Step 8: Found 2 x 2 Blocks",
    "The final Mul intersects top-pair and bottom-pair maps. Every blue cell is a found 2 x 2 block.",
    single_channel_to_grid(activations["block"], height=8, width=8),
    "Mul",
)

In [ ]:
display(html_table(
    ["hit #", "top-left row", "top-left col"],
    [[i, int(r), int(c)] for i, (r, c) in enumerate(hit_positions, start=1)]
))

In [ ]:
show_count_stage(
    count_value,
    single_channel_to_grid(activations["c1_vals"], height=1, width=5),
)

In [ ]:
show_stage(
    "Step 10: ONNX Row Tensor",
    "Foreground and background channels are concatenated into a 10-channel 1 x 5 tensor.",
    tensor_to_grid(activations["out_1x5"]),
    "Concat",
)

In [ ]:
show_stage(
    "Step 11: Final ONNX Output",
    "Pad places the 1 x 5 answer back into the fixed 30 x 30 NeuroGolf output canvas.",
    final_grid,
    "Pad",
)

In [ ]:
show_pairs(
    [{"input": final_grid, "output": selected["output"]}],
    "Left: decoded ONNX output. Right: expected target."
)

## Metric And Costs

The task metric is still exact-match over the full one-hot output tensor. A prediction passes only when the `1 x 10 x 30 x 30` output tensor matches the expected tensor.

The newest local metric used for active work is the `utils3` metric:

```python
cost = memory + params
score = max(1.0, 25.0 - log(max(1.0, cost)))
```

MACs are still profiled and checked for validity, but they are **not charged** in the score. Memory is computed from ONNX Runtime profiler output shapes with node-name sanitization, and params come from the `onnx_tool` profile.

The archive release table was built under an older accounting path, so this section shows the latest metric first and keeps the archive row only as historical reference.

In [ ]:
final_options = ort.SessionOptions()
final_options.log_severity_level = 3
final_session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=final_options,
    providers=["CPUExecutionProvider"],
)


def run_grid(grid):
    output_tensor = final_session.run(["output"], {"input": one_hot(grid)})[0]
    return tensor_to_grid(output_tensor)

metric_rows = []
total_right = 0
total_count = 0
for split in ["train", "test", "arc-gen"]:
    right = 0
    for pair in examples[split]:
        right += int(run_grid(pair["input"]) == pair["output"])
    total_right += right
    total_count += len(examples[split])
    metric_rows.append([split, right, len(examples[split]), f"{right / len(examples[split]):.1%}"])
metric_rows.append(["total", total_right, total_count, f"{total_right / total_count:.1%}"])

display(html_table(["split", "exact matches", "examples", "pass rate"], metric_rows))

In [ ]:
import os
import onnx_tool


def _profile_macs_and_params(model_path):
    profiled = onnx_tool.loadmodel(str(model_path), {"verbose": False, "constant_folding": True})
    g = profiled.graph
    g.graph_reorder_nodes()
    g.shape_infer(None)
    g.profile()
    if not g.valid_profile:
        raise ValueError("onnx_tool produced an invalid profile")
    bad_ops = {"LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"}
    for node in g.nodemap.values():
        if node.op_type.upper() in bad_ops or "Sequence" in node.op_type:
            raise ValueError(f"op type {node.op_type} is not permitted")
        if node.memory < 0 or node.params < 0 or min(node.macs) < 0:
            raise ValueError("negative profile component detected")
    return int(sum(g.macs)), int(g.params)


def _latest_metric_memory_and_rows(model_path, examples):
    sanitized = onnx.load(str(model_path))
    for node in sanitized.graph.node:
        if node.output:
            node.name = node.output[0]

    onnx.checker.check_model(sanitized, full_check=True)
    graph = shape_inference.infer_shapes(sanitized, strict_mode=True).graph
    init_names = {init.name for init in graph.initializer}
    io_names = {t.name for t in list(graph.input) + list(graph.output)}
    if io_names.intersection(init_names):
        raise ValueError("graph input/output names may not overlap initializer names")
    if sanitized.functions:
        raise ValueError("ONNX functions are not permitted")
    for opset in sanitized.opset_import:
        if opset.domain not in {"", "ai.onnx"}:
            raise ValueError("custom opset domains are not permitted")

    node_outputs = {}
    tensor_dtypes = {}
    for node in graph.node:
        for attr in node.attribute:
            if attr.type in [onnx.AttributeProto.GRAPH, onnx.AttributeProto.GRAPHS]:
                raise ValueError("subgraphs are not permitted")
        node_outputs[node.name] = list(node.output)

    for item in list(graph.input) + list(graph.value_info) + list(graph.output):
        if item.type.HasField("sequence_type"):
            raise ValueError("sequence tensors are not permitted")
        if not item.type.HasField("tensor_type"):
            continue
        tensor_type = item.type.tensor_type
        if not tensor_type.HasField("shape"):
            raise ValueError("all tensor shapes must be static")
        for dim in tensor_type.shape.dim:
            if dim.HasField("dim_param") or not dim.HasField("dim_value") or dim.dim_value <= 0:
                raise ValueError("all tensor dimensions must be positive static integers")
        if item.name in ["input", "output"]:
            continue
        tensor_dtypes[item.name] = helper.tensor_dtype_to_np_dtype(tensor_type.elem_type)

    options = ort.SessionOptions()
    options.enable_profiling = True
    options.log_severity_level = 3
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(
        sanitized.SerializeToString(),
        sess_options=options,
        providers=["CPUExecutionProvider"],
    )
    for split in ["train", "test", "arc-gen"]:
        for pair in examples[split]:
            session.run(["output"], {"input": one_hot(pair["input"])})
    trace_path = session.end_profiling()

    try:
        with open(trace_path) as f:
            trace_data = json.load(f)
    finally:
        try:
            os.remove(trace_path)
        except OSError:
            pass

    memory = 0
    current_memory = 0
    first_node_name = None
    max_rows = {}
    for event in trace_data:
        if event.get("cat") != "Node" or "args" not in event:
            continue
        if "output_type_shape" not in event["args"]:
            continue
        node_name = event.get("name", "").replace("_kernel_time", "")
        if first_node_name is None:
            first_node_name = node_name
        elif node_name == first_node_name:
            memory = max(memory, current_memory)
            current_memory = 0
        if node_name not in node_outputs:
            continue
        for i, shape_dict in enumerate(event["args"]["output_type_shape"]):
            if i >= len(node_outputs[node_name]):
                continue
            output_name = node_outputs[node_name][i]
            if output_name not in tensor_dtypes:
                continue
            dtype = np.dtype(tensor_dtypes[output_name])
            for dims in shape_dict.values():
                bytes_ = int(dtype.itemsize * math.prod(dims))
                current_memory += bytes_
                old = max_rows.get(output_name)
                if old is None or bytes_ > old[3]:
                    max_rows[output_name] = [node_name, output_name, " x ".join(map(str, dims)), bytes_]

    memory = max(memory, current_memory)
    rows = sorted(max_rows.values(), key=lambda row: (row[0], row[1]))
    return int(memory), rows


macs, latest_memory, latest_params = None, None, None
macs, latest_params = _profile_macs_and_params(ONNX_PATH)
latest_memory, latest_rows = _latest_metric_memory_and_rows(ONNX_PATH, examples)
latest_cost = latest_memory + latest_params
latest_score = max(1.0, 25.0 - math.log(max(1.0, latest_cost)))

summary_rows = [
    ["latest metric memory", latest_memory],
    ["latest metric params", latest_params],
    ["latest metric cost", latest_cost],
    ["latest metric score", f"{latest_score:.6f}"],
    ["MACs profiled but not charged", macs],
]

if release_row:
    summary_rows.extend([
        ["archive release-table cost", release_row.get("cost", "")],
        ["archive release-table score", release_row.get("score_points", "")],
        ["archive family label", release_row.get("primary_family", "")],
    ])

display(Markdown("### Latest Metric Summary"))
display(html_table(["item", "value"], summary_rows))
display(Markdown("### ORT-Profiled Intermediate Tensors"))
display(html_table(["node", "tensor", "runtime shape", "bytes"], latest_rows))

## Takeaway

`task038` is a nice example of an ONNX graph doing a visual search. Four shifted slices turn a 2 x 2 pattern search into aligned tensors; multiplication acts like logical AND; `ReduceSum` turns the hit map into a count; and the last few ops encode that count as the task's output row.